In [ ]:
import sys
from pathlib import Path

scripts_dir = Path.cwd().parent / "scripts"
sys.path.insert(0, str(scripts_dir))

In [ ]:
# Path to test .mrxs
path_CMU1 = r"E:\Christine\testdata\CMU-1.mrxs"
path_CMU3 = r"E:\Christine\testdata\CMU-3.mrxs"
zarr_dir = r"E:\Christine\testdata\zarr"
cache_tissue = r"E:\Christine\testdata\cache_tissue_artifact.pkl"
cache_features = r"E:\Christine\testdata\feature_summary.csv"

slides = [path_CMU1, path_CMU3]

In [ ]:
from tissue_artifact_segmentation import SegmentMany

segmenter = SegmentMany(slides, cache_tissue, zarr_dir, "tissue", version="default")

In [ ]:
# Visualize Single Slide
import os
from wsidata import open_wsi

zarr_path = os.path.join(zarr_dir, os.path.basename(path_CMU3).replace(".mrxs", ".zarr"))
wsi = open_wsi(path_CMU3, zarr_path)
wsi

In [ ]:
from feature_extraction import ExtractMany

model = "h-optimus-0"
extractor = ExtractMany(slides, cache_features, zarr_dir, model, remove_artifacts = False)

In [ ]:
# Generate df that resembles pathology df in 100k project
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "filename": [str(x) for x in slides],
    "class":  np.random.randint(0, 2, size=len(slides))
}).reset_index(drop=True)

times = 10
df_expanded = pd.DataFrame(np.repeat(df.values, times, axis=0), columns=df.columns)

print(df_expanded)

In [ ]:
from abmil import TrainABMILPipeline

abmil_path = r"E:\Christine\testdata\abmil.pt"

pipeline = TrainABMILPipeline(df_expanded, 'filename', 'class', 'features_h-optimus-0', 'tiles_224', zarr_dir, abmil_path)
pipeline.run_pipeline(max_tiles=50000, n_epochs=10, seed=42, validation_fraction=0.10, early_stopping_patience=5)

In [ ]:
from abmil import ABMILInference

abmil_path = r"E:\Christine\testdata\abmil.pt"
inference_path = r"E:\Christine\testdata\inference.pkl"

inference = ABMILInference(checkpoint_path=abmil_path, zarr_dir=zarr_dir, slides=slides, cache_path=inference_path)
inference.process_slides(validate = True)

In [ ]:
from roi_selection import ROISelector

selector = ROISelector(cache_path = inference_path, slide_path = path_CMU1, top_k = 10, bottom_k = 5)
selector.tiles_to_cut()

In [ ]:
selector.zoomed_view()

In [ ]:
sdata = selector.get_sdata()

top_polygons, bottom_polygons = selector.napari_polygons()

top_tiles = selector.get_tiles_gdf(top=True)
bottom_tiles = selector.get_tiles_gdf(top=False)

In [ ]:
# Find name of overview image in sdata to plot in napari
sdata


In [ ]:
import napari

viewer = napari.Viewer()

viewer.add_image(
    sdata.images["image"].data,
    channel_axis=0,
    name="WSI_preview",
)
viewer.add_shapes(
    top_polygons,
    shape_type="polygon",
    edge_color="red",
    face_color="red",
)
viewer.add_shapes(
    bottom_polygons,
    shape_type="polygon",
    edge_color="blue",
    face_color="blue",
)
napari.run()

In [ ]:
# Add manually selected calibration points to sdata.points["calibration_points"]

from spatialdata.models import PointsModel
import numpy as np

points_layer = viewer.layers["calibration_points"]
image_points = points_layer.data
print(image_points)

sdata.points["calibration_points"] = PointsModel.parse(
    np.array(image_points)
)

In [ ]:
# Add square to ensure correct placement of polygons

from spatialdata.models import ShapesModel
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np

square_layer = viewer.layers["polygons"]
image_square = square_layer.data
print(image_square)

polygons = [Polygon(coords) for coords in image_square]
square_gdf = gpd.GeoDataFrame(
    {"shape_id": [f"square_{i}" for i in range(len(polygons))]},
    geometry=polygons
)

sdata.shapes["square"] = ShapesModel.parse(square_gdf)

In [ ]:
sdata

In [ ]:
H = sdata.images["image"].data.shape[1]
print(H)

In [ ]:
from dvpio.write import write_lmd

path_lmd = os.path.join(lmd_dir, "lung5a.xml")

# Transform coordinates from napari to LMD coordinate system
affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, H],
    [0,  0, 1]
])

# Write LMD file with tiles and calibration points
write_lmd(
    path_lmd,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation
)